In [1]:
# Import required libraries
import praw
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

# Download NLTK resources
nltk.download('vader_lexicon')
nltk.download('stopwords')

# Reddit API credentials
CLIENT_ID = 'CA7MbbDocJywTqCs19M8Ig'
CLIENT_SECRET = 'vjMYDdop_TJRlHfXlH1TJPfMQTTiiQ'
USER_AGENT = 'AppName/Version by /u/AchrafLaabidi'

# Predefined lists
countries_list = [
    'Algeria', 'Angola', 'Benin', 'Botswana', 'Burkina Faso', 'Burundi', 'Cameroon',
    'Cape Verde', 'Central African Republic', 'Chad', 'Comoros', 'Congo',
    'Democratic Republic of the Congo', 'Djibouti', 'Egypt', 'Equatorial Guinea',
    'Eswatini', 'Ethiopia', 'Gabon', 'Gambia', 'Ghana', 'Guinea', 'Guinea-Bissau',
    'Kenya', 'Lesotho', 'Liberia', 'Libya', 'Madagascar', 'Malawi', 'Mali',
    'Mauritania', 'Mauritius', 'Morocco', 'Mozambique', 'Namibia', 'Niger',
    'Nigeria', 'Rwanda', 'São Tomé and Príncipe', 'Senegal', 'Seychelles',
    'Sierra Leone', 'South Africa', 'Sudan', 'Tanzania', 'Togo', 'Tunisia',
    'Uganda', 'Zambia', 'Zimbabwe'
]




sectors_keywords = {
    "Agriculture": [
        "agribusiness", "sustainable development", "economic growth", "agriculture", 
        "food security", "irrigation", "farming", "crop production", "precision agriculture"
    ],
    "Energy": [
        "renewable energy", "green energy", "energy transition", "climate change", 
        "energy efficiency", "clean energy", "solar power", "wind energy", "bioenergy"
    ],
    "Technology": [
        "digital transformation", "startup", "technological innovation", "smart cities", 
        "fintech", "mobile tech", "artificial intelligence", "IoT", "big data"
    ]
}


# Stopwords for cleaning
stop_words = set(stopwords.words('english'))

# Function to clean text
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^\w\s]', '', text)  # Remove special characters
    text = text.lower()  # Convert to lowercase
    words = text.split()
    words = [word for word in words if word not in stop_words]  # Remove stopwords
    return ' '.join(words)

# Fetch and process Reddit posts
def fetch_and_process_reddit_data(countries, keywords):
    reddit = praw.Reddit(client_id=CLIENT_ID, client_secret=CLIENT_SECRET, user_agent=USER_AGENT)
    sid = SentimentIntensityAnalyzer()

    posts_data = []
    for country in countries:
        for keyword in keywords:
            query = f"{country} {keyword}"
            for submission in reddit.subreddit('all').search(query, limit=100, time_filter='all'):
                # Clean content
                raw_content = submission.selftext or ""
                clean_content = clean_text(raw_content)

                # Skip posts with no relevant content after cleaning
                if not clean_content.strip():
                    continue

                # Analyze sentiment
                sentiment_score = sid.polarity_scores(clean_content)['compound']
                sentiment_label = 'neutral'
                if sentiment_score > 0:
                    sentiment_label = 'positive'
                elif sentiment_score < 0:
                    sentiment_label = 'negative'

                # Add data to list
                posts_data.append({
                    "country": country,
                    "query": query,
                    "clean_content": clean_content,
                    "sentiment": sentiment_label,
                    "score": submission.score,
                    "timestamp": submission.created_utc
                })

    return pd.DataFrame(posts_data)

# Function to display options for the user to select from a predefined list
def select_from_list(options, prompt):
    print(f"\n{prompt}")
    for i, option in enumerate(options, start=1):
        print(f"{i}. {option}")
    
    selected_indices = input("\nEnter the numbers of the selected options, separated by commas (e.g., 1,3,5): ")
    selected_indices = [int(i.strip()) for i in selected_indices.split(',')]

    return [options[i-1] for i in selected_indices]

def select_sector_and_keywords():
    sectors = list(sectors_keywords.keys())
    selected_sectors = select_from_list(sectors, "Select the sector you're interested in:")
    
    selected_keywords = []
    for sector in selected_sectors:
        keywords = sectors_keywords[sector]
        print(f"\nYou selected the '{sector}' sector. Now, select keywords from the following list:")
        sector_keywords = select_from_list(keywords, f"Select the keywords you're interested in under '{sector}':")
        selected_keywords.extend(sector_keywords)
    
    return selected_sectors, selected_keywords


def calculate_weighted_sentiment(posts_df):
    """
    Calculate the weighted average sentiment for each country based on post scores.

    Parameters:
    - posts_df (DataFrame): The DataFrame containing the posts.

    Returns:
    - DataFrame: A DataFrame with countries and their weighted average sentiment.
    """
    # Map sentiment labels to numerical values
    sentiment_mapping = {"positive": 1, "neutral": 0, "negative": -1}
    posts_df["sentiment_value"] = posts_df["sentiment"].map(sentiment_mapping)

    # Group by country and calculate weighted average
    weighted_sentiment = posts_df.groupby(["country", "query"]).apply(
        lambda group: (
            (group["score"] * group["sentiment_value"]).sum() / group["score"].sum()
            if group["score"].sum() > 0 else 0
        )
    ).reset_index(name="weighted_sentiment")

    return weighted_sentiment



country = input("Enter the country name (or type 'all' to select multiple countries): ")

# Let the user select countries if they want to select more than one
if country.lower() == 'all':
    selected_countries = select_from_list(countries_list, "Select the countries you're interested in:")
else:
    selected_countries = [country.strip()]

# Let the user select keywords from the predefined list
selected_sectors, selected_keywords = select_sector_and_keywords()


# Fetch and process data
reddit_data = fetch_and_process_reddit_data(selected_countries, selected_keywords)

# Check if data is available
if reddit_data.empty:
    print("No data found for the selected options.")
    
else:
    # Save the data to a CSV file
    weighted_sentiment_df = calculate_weighted_sentiment(reddit_data)
    print("\nWeighted sentiment per country:")
    print(weighted_sentiment_df)
    output_file = f"reddit_posts_{'_'.join(selected_countries).replace(' ', '_')}_{'_'.join(selected_keywords).replace(' ', '_')}.csv"

    reddit_data.to_csv(output_file, index=False)
    print(f"Data collection and processing completed. Results saved to '{output_file}'.")
    
        


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\dell\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dell\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Enter the country name (or type 'all' to select multiple countries):  all



Select the countries you're interested in:
1. Algeria
2. Angola
3. Benin
4. Botswana
5. Burkina Faso
6. Burundi
7. Cameroon
8. Cape Verde
9. Central African Republic
10. Chad
11. Comoros
12. Congo
13. Democratic Republic of the Congo
14. Djibouti
15. Egypt
16. Equatorial Guinea
17. Eswatini
18. Ethiopia
19. Gabon
20. Gambia
21. Ghana
22. Guinea
23. Guinea-Bissau
24. Kenya
25. Lesotho
26. Liberia
27. Libya
28. Madagascar
29. Malawi
30. Mali
31. Mauritania
32. Mauritius
33. Morocco
34. Mozambique
35. Namibia
36. Niger
37. Nigeria
38. Rwanda
39. São Tomé and Príncipe
40. Senegal
41. Seychelles
42. Sierra Leone
43. South Africa
44. Sudan
45. Tanzania
46. Togo
47. Tunisia
48. Uganda
49. Zambia
50. Zimbabwe



Enter the numbers of the selected options, separated by commas (e.g., 1,3,5):  4,7



Select the sector you're interested in:
1. Agriculture
2. Energy
3. Technology



Enter the numbers of the selected options, separated by commas (e.g., 1,3,5):  1



You selected the 'Agriculture' sector. Now, select keywords from the following list:

Select the keywords you're interested in under 'Agriculture':
1. agribusiness
2. sustainable development
3. economic growth
4. agriculture
5. food security
6. irrigation
7. farming
8. crop production
9. precision agriculture



Enter the numbers of the selected options, separated by commas (e.g., 1,3,5):  8



Weighted sentiment per country:
    country                     query  weighted_sentiment
0  Botswana  Botswana crop production            0.824993
1  Cameroon  Cameroon crop production            0.791211
Data collection and processing completed. Results saved to 'reddit_posts_Botswana_Cameroon_crop_production.csv'.


In [24]:
import gensim.downloader as api

# Download the model and save it to a local file
model = api.load("word2vec-google-news-300")

# Save the model to a file
model.save("word2vec-google-news-300.model")




In [52]:
# Function to get related words using Word2Vec
def get_related_words(word, top_n=10):
    try:
        # Get similar words based on cosine similarity
        similar_words = model.most_similar(word, topn=top_n)
        return [word for word, _ in similar_words]
    except KeyError:
        return f"'{word}' not in vocabulary."

# Example usage
word = "Climate-Change"
related_words = get_related_words(word)
print(f"Related words to '{word}':")
print(related_words)

Related words to 'Climate-Change':
['Climate-change', 'Climate', 'Climate-gate', 'ClimateGate', 'ClimateAudit', 'Climatic', 'Cap-and-Trade', 'Global-warming', 'Climactic', 'Climates']
